In [ ]:
from validate import validation_metrics
from scipy import stats
import matplotlib
import glob
from scipy.stats import entropy
from scipy import integrate
from scipy.signal import butter, lfilter, freqz, filtfilt


import os
import pandas as pd
import numpy as np
import torch
from torchvision import transforms

from utils.utils import load_config, create_folder, scale_coords
from utils.plots import *
from dataloaders import *
from models import *
from validate import *

from utils.plots import *
from utils.utils import scale_coords, resize_landmarks


import pickle
import cv2

from insightface.app import FaceAnalysis
from tqdm import tqdm

from XAI import *

from tqdm import tqdm
import re
import gc

from torch.utils.data import DataLoader, TensorDataset

import matplotlib

import warnings
warnings.filterwarnings("ignore")

In [ ]:
path_icopevid = 'Datasets\\Originais\\iCOPE\\iCOPEvid'
path_icopevid_frames = 'Datasets\\Originais\\iCOPE\\iCOPEvid\\all_frames'

In [ ]:

def butter_lowpass_filter(data, cutoff=2, fs=30, order=2):
    normal_cutoff = cutoff / (0.5 * fs)
    # Get the filter coefficients 
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

def moving_average(x, w):
    return np.convolve(x, np.ones(w), 'valid') / w

def get_hist(signal):
    return np.histogram(signal, bins=np.linspace(0.0, 1.0, 11))

def get_probs(signal):
    hist, _ = get_hist(signal)
    return hist / len(signal)

def get_entropy(signal):
    pk = get_probs(signal)
    return entropy(pk, base=2)

def get_entropy_curve(signal):
    hist, bin_edges = get_hist(signal)
    pk = hist / len(signal)
    entropies = []
    for x in signal:
        idx = np.digitize(x, bin_edges, right=True)
        pkx = pk[idx-1]
        entropies.append(-((pkx * np.log(pkx)) / np.log(2)))
    return entropies

def get_auc(signal):
    return integrate.trapezoid(signal) / len(signal)

def get_auc_curve(signal):
    window_size = 10
    step = 1
    auc_curve = []
    for x in range(0, len(signal)-window_size, step):
        window = signal[x:x+window_size]
        auc_curve.append(get_auc(window))
    return auc_curve

def interp_curve(signal):
    time = np.linspace(0,len(signal),len(signal))
    # Find indices of missing values
    missing_indices = np.isnan(signal)
    # Find indices of non-missing values
    valid_indices = ~missing_indices
    # Perform linear interpolation
    interpolated_values = np.interp(time[missing_indices], time[valid_indices], signal[valid_indices])
    # Replace missing values with interpolated values
    signal[missing_indices] = interpolated_values

    return signal

def fill_nan(signal):
    # Find indices of missing values
    missing_indices = np.isnan(signal)
    # Find indices of non-missing values
    valid_indices = ~missing_indices
    # Perform linear interpolation
    mean = np.mean(signal[valid_indices])
    # Replace missing values with interpolated values
    signal[missing_indices] = mean

    return signal


def peak_to_peak(signal):
    return (np.abs(np.max(signal)) - np.abs(np.min(signal)))

def derivative(signal, power=False):
    if power:
        return np.power(np.diff(signal),2)
    else:
        return np.diff(signal)
    
def RMS(signal):
    return np.sqrt(np.mean(np.square(signal)))

def thresh_crossing(signal, thresh=0.5):
    return len(np.where(np.diff(np.sign(signal-thresh)))[0]) 

In [ ]:
model_name = "NCNN_FINAL"
path_experiments = 'experiments\\' + model_name

with open(os.path.join(path_experiments,'icopevid',f'results_icopevid_MCDP_50_0.3.pkl'), 'rb') as f:
    results_video = pickle.load(f)


mcdp = True

In [ ]:
from __future__ import annotations

import os
import pickle
import gc
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple, Union

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from XAI.post_processing import kmeans_post_processing
plt.style.use('utils\plotstyle.mplstyle')


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class PainSignThresholds:
    """Operational limits.
    theta_1: decision threshold (pain vs no pain)
    theta_2_low/theta_2_high: precision/ambiguity bounds for probability
    theta_3: uncertainty bound (e.g., std from MCDropout or ensemble disagreement)
    """
    theta_1: float
    theta_3: float
    theta_2_low: float = 0.2
    theta_2_high: float = 0.8


def get_thresholds_for_model(model_name: str) -> PainSignThresholds:
    # Keep your current values, but isolate them here.
    if "NCNN" in model_name:
        return PainSignThresholds(theta_1=0.4551, theta_3=0.1186, theta_2_low=0.2, theta_2_high=0.8)
    if "VGGFace" in model_name:
        return PainSignThresholds(theta_1=0.5013, theta_3=0.0621, theta_2_low=0.2, theta_2_high=0.8)
    return PainSignThresholds(theta_1=0.4743, theta_3=0.0130, theta_2_low=0.2, theta_2_high=0.8)


@dataclass(frozen=True)
class PlotStyle:
    # Palette aligned with clinical semantics.
    signal_color: str = "#1C1C1C"       # Black Ink
    certain_color: str = "#4CB5AE"      # Confidence Teal
    uncertain_color: str = "#6A4C93"    # Uncertainty Purple
    ci_color: str = "#6A4C93"           # Uncertainty Purple
    grid_color: str = "#9E9E9E"         # Gray Neutral
    ci_alpha: float = 0.18

    # Background bands
    band_no_pain_color: str = "#2E86AB"     # Comfort Blue
    band_pain_color: str = "#D72638"        # Pain Red
    band_ambiguous_color: str = "#F29E4C"   # Alert Orange
    band_ambiguous_alpha: float = 0.0
    band_no_pain_alpha: float = 0.0
    band_pain_alpha: float = 0.0

    # Typography
    title_size: int = 14
    label_size: int = 12
    tick_size: int = 11


REGION_COLOR_MAP = {
    "eyes": "#1f77b4",
    "eyebrowns": "#9467bd",
    "cheeks": "#ff7f0e",
    "nose": "#2ca02c",
    "mouth": "#d62728",
    "chin": "#8c564b",
    "forehead": "#e377c2",
    "between_eyes": "#7f7f7f",
    "nasolabial_folds": "#bcbd22",
    "outside": "#17becf",
}

# ---------------------------------------------------------------------
# I/O helpers (frames + XAI strips)
# ---------------------------------------------------------------------

cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ["green", "yellow", "red"])


def _safe_read_rgb(path: Path, size: Tuple[int, int]) -> np.ndarray:
    img = cv2.imread(str(path))
    if img is None:
        return np.zeros((size[1], size[0], 3), dtype=np.float32)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0

def _safe_read_XAI_mask(path: Path, size: Tuple[int, int]) -> np.ndarray:
    if not path.exists():
        return np.zeros((size[1], size[0], 3), dtype=np.float32)
    if "_blank" in path.name:
        return np.zeros((size[1], size[0], 3), dtype=np.float32)
    data = np.load(path)
    mask = data['mask_raw']
    mask, alpha = kmeans_post_processing(mask)
    mask = cv2.resize(mask, size, interpolation=cv2.INTER_AREA)
    alpha = cv2.resize(alpha, size, interpolation=cv2.INTER_AREA)
    mask = np.clip(mask, 0.0, 1.0)
    alpha = np.clip(alpha, 0.0, 1.0)
    colored = cmap(mask)[..., :3].astype(np.float32)
    colored *= alpha[..., None].astype(np.float32)
    return colored

def load_video_strip(
    video_dir: Union[str, Path],
    model_name: str,
    xai_root: Union[str, Path],
    frame_step: int = 30,
    out_size: Tuple[int, int] = (256, 256),
    suffix: str = ".jpg",
) -> np.ndarray:
    """Returns a vertical strip: [frames; frames+merged XAI] where each row is a horizontal concatenation."""
    video_dir = Path(video_dir)
    xai_root = Path(xai_root)

    # IMPORTANT: os.listdir() is not guaranteed sorted; this matters for time coherence.
    img_files = sorted([p for p in video_dir.iterdir() if p.suffix.lower() == suffix])

    frames: List[np.ndarray] = []
    overlays: List[np.ndarray] = []

    merged_dir = xai_root / video_dir.name / "MERGED_MASKS"

    for i in range(0, len(img_files), frame_step):
        frame_path = img_files[i]
        frame = _safe_read_rgb(frame_path, out_size)
        frames.append(frame)

        # Masks are expected to have the same stem inside MERGED_MASKS.
        mask_path = merged_dir / f"{frame_path.stem}.npz"
        mask = _safe_read_XAI_mask(mask_path, out_size)
        alpha = mask.max(axis=2, keepdims=True)
        overlay = frame * (1.0 - alpha) + mask
        overlays.append(np.clip(overlay, 0.0, 1.0))

    if len(frames) == 0:
        # Return an empty placeholder strip
        h, w = out_size[1], out_size[0]
        return np.zeros((h * 2, w, 3), dtype=np.float32)

    frame_row = np.hstack(frames)
    overlay_row = np.hstack(overlays)
    return np.vstack([frame_row, overlay_row])


# ---------------------------------------------------------------------
# XAI region tracking (per-frame)
# ---------------------------------------------------------------------

METHOD_VIZU = "absolute"  # 'absolute', 'positive', 'negative', 'all'

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

from XAI.metrics import create_face_regions_masks, calculate_xai_score

def _zero_region_scores(region_names: Optional[Iterable[str]] = None) -> Dict[str, float]:
    if region_names is None:
        region_names = list(REGION_COLOR_MAP.keys())
    return {name: 0.0 for name in region_names}


def merge_symmetric_masks(face_masks):
    merge_map = {
        ("left_eye", "right_eye"): "eyes",
        ("left_cheek", "right_cheek"): "cheeks",
        ("left_eyebrown", "right_eyebrown"): "eyebrowns",
        ("left_nasolabial_fold", "right_nasolabial_fold"): "nasolabial_folds",
    }

    new_masks = {}
    used_keys = set()

    for (left, right), new_key in merge_map.items():
        if left in face_masks and right in face_masks:
            new_masks[new_key] = np.logical_or(face_masks[left], face_masks[right]).astype(np.uint8)
            used_keys.update([left, right])

    for key, mask in face_masks.items():
        if key not in used_keys:
            new_masks[key] = mask

    return new_masks


def _load_xai_raw_mask(mask_path: Path, mask_key: str = "mask_raw") -> np.ndarray | None:
    if not mask_path.exists():
        return None
    if "_blank" in mask_path.name:
        return None
    with np.load(mask_path) as data:
        if mask_key not in data:
            return None
        return data[mask_key]


def extract_region_scores_video(
    video_dir: Union[str, Path],
    xai_root: Union[str, Path],
    *,
    landmark_dir: Union[str, Path] | None = None,
    frame_step: int = 1,
    mask_size: Tuple[int, int] = (512, 512),
    suffix: str = ".jpg",
    mask_key: str = "mask_raw",
    method_vizu: str = METHOD_VIZU,
    fps: Optional[float] = None,
    duration_s: Optional[float] = None,
) -> pd.DataFrame:
    """Return per-frame region scores for a video.

    Output columns: frame, frame_idx, time_s + one column per face region.
    If landmark_dir is None, defaults to <video_dir>/landmarks.
    """
    video_dir = Path(video_dir)
    xai_root = Path(xai_root)
    if landmark_dir is None:
        landmark_dir = video_dir / "landmarks"
    else:
        landmark_dir = Path(landmark_dir)

    img_files = sorted([p for p in video_dir.iterdir() if p.suffix.lower() == suffix])
    merged_dir = xai_root / video_dir.name / "MERGED_MASKS"

    rows = []
    for i in tqdm(range(0, len(img_files), frame_step), desc=f"Frames {video_dir.name}"):
        frame_path = img_files[i]
        mesh_path = landmark_dir / f"{frame_path.stem}.pkl"
        if not mesh_path.exists():
            row = {
                "frame": frame_path.stem,
                "frame_idx": i,
            }
            row.update(_zero_region_scores())
            rows.append(row)
            continue

        mask_path = merged_dir / f"{frame_path.stem}.npz"
        mask = _load_xai_raw_mask(mask_path, mask_key=mask_key)
        if mask is None:
            mask = np.zeros((512, 512), dtype=np.float32)

        mask = np.nan_to_num(mask, copy=False)
        mask_resized = cv2.resize(mask, mask_size, interpolation=cv2.INTER_AREA)
        mask_resized, alpha = kmeans_post_processing(mask_resized)
        mask_resized = mask_resized * alpha
        

        with mesh_path.open("rb") as f:
            mesh = np.array(pickle.load(f))
        regions = merge_symmetric_masks(create_face_regions_masks(mesh))

        region_scores = calculate_xai_score(mask_resized, regions, sort=False)
        total_score = float(np.sum(list(region_scores.values())))
        if total_score > 0:
            region_scores = {k: v / total_score for k, v in region_scores.items()}

        row = {
            "frame": frame_path.stem,
            "frame_idx": i,
        }
        row.update(region_scores)
        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    if fps is not None:
        df["time_s"] = df["frame_idx"] / float(fps)
    elif duration_s is not None:
        df["time_s"] = np.linspace(0, float(duration_s), len(df))
    else:
        df["time_s"] = df["frame_idx"].astype(float)

    return df


def _moving_average_1d(x: np.ndarray, window: int) -> np.ndarray:
    if window <= 1:
        return x
    kernel = np.ones(window, dtype=float) / float(window)
    return np.convolve(x, kernel, mode="valid")

# ---------------------------------------------------------------------
# Pain sign computation
# ---------------------------------------------------------------------

@dataclass
class PainSignResult:
    time_s: np.ndarray              # shape [T]
    p_hat: np.ndarray               # shape [T]
    sigma_hat: Optional[np.ndarray] # shape [T] or None if not available
    p_summary: float
    sigma_summary: Optional[float]
    pred_label: int                 # 0/1 based on theta_1


def compute_pain_sign(
    video_results: Dict,
    *,
    mcdp: bool,
    theta_1: float,
    ma_window: int = 30,
    duration_s: float = 20.0,
) -> PainSignResult:
    """Computes smoothed probability and (optionally) smoothed uncertainty curve.

    Expects existing functions in your codebase:
      - interp_curve(x: np.ndarray) -> np.ndarray
      - moving_average(x: np.ndarray, window: int) -> np.ndarray
    """
    if mcdp:
        mean_probs: List[float] = []
        std_probs: List[float] = []
        for fold in list(video_results.keys()):
            for arr in video_results[fold]:
                arr = np.asarray(arr)
                if np.isnan(arr).any():
                    mean_probs.append(np.nan)
                    std_probs.append(np.nan)
                else:
                    mean_probs.append(float(arr.mean()))
                    std_probs.append(float(arr.std()))

        mean_probs = interp_curve(np.asarray(mean_probs))
        std_probs = interp_curve(np.asarray(std_probs))

        p_hat = _moving_average_1d(mean_probs, ma_window)
        sigma_hat = _moving_average_1d(std_probs, ma_window)

    else:
        stacked: List[np.ndarray] = []
        for fold in list(video_results.keys()):
            stacked.append(interp_curve(video_results[fold]))
        stacked = np.asarray(stacked)
        mean_probs = stacked.mean(axis=0)

        p_hat = _moving_average_1d(mean_probs, ma_window)
        sigma_hat = None

    time_s = np.linspace(0, duration_s, len(p_hat))
    p_summary = float(np.mean(p_hat))
    sigma_summary = float(np.mean(sigma_hat)) if sigma_hat is not None else None
    pred_label = int(p_summary >= theta_1)

    return PainSignResult(
        time_s=time_s,
        p_hat=p_hat,
        sigma_hat=sigma_hat,
        p_summary=p_summary,
        sigma_summary=sigma_summary,
        pred_label=pred_label,
    )


# ---------------------------------------------------------------------
# Visualization (doctor-facing)
# ---------------------------------------------------------------------

def _segments_from_mask(mask: np.ndarray) -> List[Tuple[int, int]]:
    """Convert boolean mask into contiguous [start, end) index segments."""
    if mask.size == 0:
        return []
    m = mask.astype(np.int8)
    changes = np.diff(m, prepend=m[0])
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]
    if m[0] == 1:
        starts = np.r_[0, starts]
    if m[-1] == 1:
        ends = np.r_[ends, len(m)]
    return list(zip(starts, ends))


def plot_pain_sign(
    *,
    video_name: str,
    strip_img: np.ndarray,
    ps: PainSignResult,
    thresholds: PainSignThresholds,
    true_label: int,
    model_name: str,
    save_path: Union[str, Path],
    region_df: Optional[pd.DataFrame] = None,
    region_top_k: int = 0,
    region_selection: str = "mean",
    region_smooth_window: int = 30,
    style: PlotStyle = PlotStyle(),
) -> None:
    """Clinical plot:
    - Probability curve with uncertainty ribbon
    - Background bands showing:
        - no pain region (below theta_1)
        - pain region (above theta_1)
        - ambiguous probability zone between theta_2_low and theta_2_high
    - Segments colored by uncertainty (sigma <= theta_3 = certain; else uncertain)
    - Simple summary box (decision + reliability)
    - Region-importance curves (top regions) under pain curve
    - Frame/XAI strip aligned underneath
    """
    time = ps.time_s
    p = ps.p_hat
    sigma = ps.sigma_hat

    has_regions = region_df is not None and not region_df.empty

    if has_regions:
        fig = plt.figure(figsize=(16, 9))
        gs = GridSpec(nrows=3, ncols=1, height_ratios=[3.0, 1.8, 2.0], hspace=0.12)
    else:
        fig = plt.figure(figsize=(16, 7))
        gs = GridSpec(nrows=2, ncols=1, height_ratios=[3.2, 2.0], hspace=0.12)

    ax = fig.add_subplot(gs[0])

    # Background: pain/no-pain bands relative to theta_1 (subtle, not distracting)
    ax.axhspan(0, thresholds.theta_1, alpha=style.band_no_pain_alpha, color=style.band_no_pain_color, lw=0)
    ax.axhspan(thresholds.theta_1, 1, alpha=style.band_pain_alpha, color=style.band_pain_color, lw=0)

    # Background: ambiguous probability zone (theta_2 band)
    ax.axhspan(thresholds.theta_2_low, thresholds.theta_2_high, alpha=style.band_ambiguous_alpha, color=style.band_ambiguous_color, lw=0, label="theta_2 band")

    # Uncertainty ribbon (if available)
    if sigma is not None:
        upper = np.clip(p + sigma, 0, 1)
        lower = np.clip(p - sigma, 0, 1)
        ax.fill_between(time, lower, upper, color=style.ci_color, alpha=style.ci_alpha, edgecolor="none", label="Uncertainty band (±σ)")

    # Determine certainty mask from theta_3 (if available)
    if sigma is not None:
        certain = sigma <= thresholds.theta_3
    else:
        # If no sigma curve exists, treat all as "certain" for plotting.
        certain = np.ones_like(p, dtype=bool)

    # Probability precision (theta_2) mask
    # IMPORTANT: your original idx_precise was impossible ((<=low) & (>=high)).
    # Here: "precise" = outside ambiguous zone; "ambiguous" = inside [low, high].
    ambiguous = (p >= thresholds.theta_2_low) & (p <= thresholds.theta_2_high)
    precise = ~ambiguous

    # Plot curve in segments: certain vs uncertain
    certain_segs = _segments_from_mask(certain)
    uncertain_segs = _segments_from_mask(~certain)

    for s, e in certain_segs:
        ax.plot(time[s:e], p[s:e], color=style.certain_color, lw=2.2, solid_capstyle="round")
    for s, e in uncertain_segs:
        ax.plot(time[s:e], p[s:e], color=style.uncertain_color, lw=2.6, solid_capstyle="round")

    # Threshold lines
    ax.axhline(thresholds.theta_1, linestyle=":", color=style.grid_color, lw=1.6)
    ax.text(time.max() + 0.2, thresholds.theta_1, r"$\theta_1$", va="center", fontsize=style.tick_size)

    ax.axhline(thresholds.theta_2_low, linestyle=":", color=style.grid_color, lw=1.2)
    ax.axhline(thresholds.theta_2_high, linestyle=":", color=style.grid_color, lw=1.2)
    ax.text(time.max() + 0.2, thresholds.theta_2_low, r"$\theta_{2,low}$", va="center", fontsize=style.tick_size)
    ax.text(time.max() + 0.2, thresholds.theta_2_high, r"$\theta_{2,high}$", va="center", fontsize=style.tick_size)

    # Mark theta_1 crossings (inflexion points)
    idx_cross = np.where(np.diff((p >= thresholds.theta_1).astype(int)) != 0)[0]
    if idx_cross.size:
        ax.scatter(time[idx_cross], np.full(idx_cross.shape, thresholds.theta_1), s=40, color=style.uncertain_color, zorder=5, label="Decision crossings")

    # Axes formatting
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlim(time.min(), time.max())
    ax.set_ylabel("Pain probability", fontsize=style.label_size)
    ax.set_xlabel("Time (s)", fontsize=style.label_size)
    if has_regions:
        ax.set_xlabel("")
        ax.tick_params(labelbottom=False)
    ax.tick_params(axis="both", labelsize=style.tick_size)
    ax.grid(True, axis="y", alpha=0.18, color=style.grid_color)
    ax.grid(False, axis="x")

    # Title + compact summary intended for clinicians
    label_txt = "Pain" if true_label == 1 else "No pain"
    pred_txt = "Pain" if ps.pred_label == 1 else "No pain"

    reliability_parts = []
    reliability_parts.append(f"Mean p̂={ps.p_summary:.3f} → {pred_txt}")
    if ps.sigma_summary is not None:
        reliability_parts.append(f"Mean σ̂={ps.sigma_summary:.3f} → {'Certain' if ps.sigma_summary <= thresholds.theta_3 else 'Uncertain'}")
    reliability_parts.append(f"Crossings={int(idx_cross.size)}")
    #reliability_parts.append(f"Ambiguous%={100.0*float(np.mean(ambiguous)):.1f}%")
    #if sigma is not None:
        #reliability_parts.append(f"Uncertain%={100.0*float(np.mean(~certain)):.1f}%")

    ax.set_title(
        f"{video_name} | Model: {model_name} | True: {label_txt} | " + " | ".join(reliability_parts),
        fontsize=style.title_size,
        pad=10,
    )

    # Legend: keep minimal
    handles = [
        plt.Line2D([0], [0], color=style.certain_color, lw=2.2, label="Probability (certain)"),
        plt.Line2D([0], [0], color=style.uncertain_color, lw=2.6, label="Probability (uncertain)"),
        #plt.Rectangle((0, 0), 1, 1, color=style.band_ambiguous_color, alpha=style.band_ambiguous_alpha, label="theta_2 band"),
    ]
    if sigma is not None:
        handles.append(plt.Rectangle((0, 0), 1, 1, color=style.ci_color, alpha=style.ci_alpha, label="±σ band"))
    ax.legend(handles=handles, loc="upper left", frameon=True, framealpha=0.9)

    # Middle: region curves (optional)
    if has_regions:
        axr = fig.add_subplot(gs[1], sharex=ax)
        region_cols = [c for c in region_df.columns if c not in ("frame", "frame_idx", "time_s")]
        if "time_s" in region_df.columns:
            time_r = region_df["time_s"].to_numpy(dtype=float)
        elif "frame_idx" in region_df.columns:
            time_r = region_df["frame_idx"].to_numpy(dtype=float)
        else:
            time_r = np.arange(len(region_df), dtype=float)

        if region_cols:
            if region_selection == "max":
                agg = region_df[region_cols].max(axis=0)
            else:
                agg = region_df[region_cols].mean(axis=0)
            agg = agg[agg > 0]

            if region_top_k is None or region_top_k <= 0:
                top_regions = agg.sort_values(ascending=False).index.tolist()
            else:
                top_regions = agg.sort_values(ascending=False).head(region_top_k).index.tolist()

            if top_regions:
                data = region_df[top_regions].to_numpy(dtype=float)
                if region_smooth_window > 1:
                    data = np.column_stack([
                        _moving_average_1d(data[:, i], region_smooth_window) for i in range(data.shape[1])
                    ])

                default_colors = plt.cm.get_cmap("tab20", len(top_regions))
                for idx, region in enumerate(top_regions):
                    color = REGION_COLOR_MAP.get(region, default_colors(idx))
                    time_r = np.linspace(0, 20, len(data[:, idx]))
                    axr.plot(time_r, data[:, idx], lw=1.8, color=color, label=region)
                axr.legend(loc="upper right", ncol=2, fontsize=9)
            else:
                axr.text(0.5, 0.5, "No positive region scores", transform=axr.transAxes, ha="center", va="center")
        else:
            axr.text(0.5, 0.5, "No region columns to plot", transform=axr.transAxes, ha="center", va="center")

        axr.set_ylabel("Region importance", fontsize=style.label_size)
        axr.set_xlabel("Time (s)", fontsize=style.label_size)
        axr.tick_params(axis="both", labelsize=style.tick_size)
        axr.grid(True, axis="y", alpha=0.18, color=style.grid_color)
        axr.set_ylim(-0.05, 1.05)
        if time_r.size:
            axr.set_xlim(time.min(), time.max())

        ax_strip = fig.add_subplot(gs[2])
    else:
        ax_strip = fig.add_subplot(gs[1])

    # Bottom: frames + XAI strip
    ax_strip.imshow(strip_img)
    ax_strip.axis("off")

    # Row labels (Frames / Frames + XAI)
    h = strip_img.shape[0]
    ax_strip.text(-0.01, 0.75, "Frames", transform=ax_strip.transAxes, rotation=90, va="center", ha="right", fontsize=style.label_size)
    ax_strip.text(-0.01, 0.25, "Frames + XAI", transform=ax_strip.transAxes, rotation=90, va="center", ha="right", fontsize=style.label_size)

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------
# End-to-end runner
# ---------------------------------------------------------------------

def run_pain_sign_report(
    results_video: Dict[str, Dict],
    *,
    model_name: str,
    path_icopevid_frames: Union[str, Path],
    xai_root: Union[str, Path],
    out_dir: Union[str, Path],
    mcdp: bool,
    ma_window: int = 30,
    duration_s: float = 20.0,
    frame_step: int = 30,
    region_frame_step: int = 1,
    include_region_curves: bool = True,
    region_top_k: int = 0,
    region_selection: str = "mean",
    region_smooth_window: int = 30,
) -> Dict[str, np.ndarray]:
    """Generates per-video PDFs and returns arrays for metrics.
    Expects existing:
      - validation_metrics(preds, probs, labels) -> dict or printable
      - get_entropy / thresh_crossing if you want to add extra summary items
    """
    thresholds = get_thresholds_for_model(model_name)
    out_dir = Path(out_dir)
    frames_root = Path(path_icopevid_frames)
    xai_root = Path(xai_root)

    labels: List[int] = []
    preds: List[int] = []
    probs: List[float] = []

    for video_name in results_video.keys():
        video_dir = frames_root / video_name
        strip = load_video_strip(video_dir, model_name=model_name, xai_root=xai_root, frame_step=frame_step)

        true_label = 1 if "Pain" in video_name else 0
        labels.append(true_label)

        ps = compute_pain_sign(
            results_video[video_name],
            mcdp=mcdp,
            theta_1=thresholds.theta_1,
            ma_window=ma_window,
            duration_s=duration_s,
        )

        preds.append(ps.pred_label)
        probs.append(ps.p_summary)

        region_df = None
        if include_region_curves:
            try:
                region_df = extract_region_scores_video(
                    video_dir=video_dir,
                    xai_root=xai_root,
                    frame_step=region_frame_step,
                    duration_s=duration_s,
                )
            except Exception as exc:
                print(f"Region curves skipped for {video_name}: {exc}")
                region_df = None

        save_path = out_dir / f"{video_name}_{model_name}.pdf"
        plot_pain_sign(
            video_name=video_name,
            strip_img=strip,
            ps=ps,
            thresholds=thresholds,
            true_label=true_label,
            model_name=model_name,
            save_path=save_path,
            region_df=region_df,
            region_top_k=region_top_k,
            region_selection=region_selection,
            region_smooth_window=region_smooth_window,
        )

    gc.collect()

    preds_arr = np.asarray(preds, dtype=int)
    probs_arr = np.asarray(probs, dtype=float)
    labels_arr = np.asarray(labels, dtype=int)

    return {
        "preds": preds_arr,
        "probs": probs_arr,
        "labels": labels_arr,
    }







In [ ]:
# ---------------------------------------------------------------------
# Animation: frame-by-frame pain sign + XAI overlay
# ---------------------------------------------------------------------

def render_pain_sign_animation(
    *,
    video_name: str,
    video_results: dict,
    model_name: str,
    video_dir: Union[str, Path],
    xai_root: Union[str, Path],
    out_path: Union[str, Path],
    mcdp: bool,
    thresholds: Optional[PainSignThresholds] = None,
    true_label: Optional[int] = None,
    fps: Optional[float] = None,
    duration_s: Optional[float] = 20.0,
    ma_window: int = 30,
    frame_step: int = 1,
    out_size: Tuple[int, int] = (256, 256),
    suffix: str = ".jpg",
    include_region_curves: bool = True,
    region_df: Optional[pd.DataFrame] = None,
    region_frame_step: int = 1,
    region_top_k: int = 0,
    region_selection: str = "mean",
    region_smooth_window: int = 30,
    mask_key: str = "mask_raw",
    figsize: Tuple[int, int] = (16, 9),
    dpi: int = 110,
    codec: str = "mp4v",
    show_progress: bool = True,
    style: PlotStyle = PlotStyle(),
) -> Path:
    """Create an MP4 animation for a single video.

    The animation mirrors plot_pain_sign, but reveals curves frame-by-frame and
    updates the frame/XAI overlay at each timestep.
    """
    if thresholds is None:
        thresholds = get_thresholds_for_model(model_name)

    video_dir = Path(video_dir)
    xai_root = Path(xai_root)
    out_path = Path(out_path)

    img_files = sorted([p for p in video_dir.iterdir() if p.suffix.lower() == suffix])
    if not img_files:
        raise ValueError(f"No frames found in {video_dir} with suffix {suffix}")

    if fps is None:
        if duration_s is None:
            fps = 30.0
            duration_s = len(img_files) / fps
        else:
            fps = max(1.0, len(img_files) / float(duration_s))

    if duration_s is None:
        duration_s = len(img_files) / fps

    ps = compute_pain_sign(
        video_results,
        mcdp=mcdp,
        theta_1=thresholds.theta_1,
        ma_window=ma_window,
        duration_s=duration_s,
    )

    if true_label is None:
        name_for_label = video_name or video_dir.name
        true_label = 1 if "Pain" in name_for_label else 0

    has_regions = include_region_curves
    if include_region_curves and region_df is None:
        try:
            region_df = extract_region_scores_video(
                video_dir=video_dir,
                xai_root=xai_root,
                frame_step=region_frame_step,
                duration_s=duration_s,
                mask_key=mask_key,
            )
        except Exception as exc:
            print(f"Region curves skipped for {video_name}: {exc}")
            region_df = None
            has_regions = False

    if region_df is not None and not region_df.empty:
        has_regions = True
    else:
        has_regions = False

    # Figure setup
    if has_regions:
        fig = plt.figure(figsize=figsize, dpi=dpi)
        gs = GridSpec(nrows=3, ncols=1, height_ratios=[3.0, 1.8, 2.0], hspace=0.12)
    else:
        fig = plt.figure(figsize=figsize, dpi=dpi)
        gs = GridSpec(nrows=2, ncols=1, height_ratios=[3.2, 2.0], hspace=0.12)

    ax = fig.add_subplot(gs[0])

    # Background bands
    ax.axhspan(0, thresholds.theta_1, alpha=style.band_no_pain_alpha, color=style.band_no_pain_color, lw=0)
    ax.axhspan(thresholds.theta_1, 1, alpha=style.band_pain_alpha, color=style.band_pain_color, lw=0)
    ax.axhspan(thresholds.theta_2_low, thresholds.theta_2_high, alpha=style.band_ambiguous_alpha,
               color=style.band_ambiguous_color, lw=0)

    time = ps.time_s
    p = ps.p_hat
    sigma = ps.sigma_hat

    if sigma is not None:
        certain = sigma <= thresholds.theta_3
    else:
        certain = np.ones_like(p, dtype=bool)

    p_certain = np.where(certain, p, np.nan)
    p_uncertain = np.where(~certain, p, np.nan)

    line_certain, = ax.plot([], [], color=style.certain_color, lw=2.2, solid_capstyle="round")
    line_uncertain, = ax.plot([], [], color=style.uncertain_color, lw=2.6, solid_capstyle="round")

    # Threshold lines and labels
    ax.axhline(thresholds.theta_1, linestyle=":", color=style.grid_color, lw=1.6)
    ax.text(time.max() + 0.2, thresholds.theta_1, r"$\theta_1$", va="center", fontsize=style.tick_size)
    ax.axhline(thresholds.theta_2_low, linestyle=":", color=style.grid_color, lw=1.2)
    ax.axhline(thresholds.theta_2_high, linestyle=":", color=style.grid_color, lw=1.2)
    ax.text(time.max() + 0.2, thresholds.theta_2_low, r"$\theta_{2,low}$", va="center", fontsize=style.tick_size)
    ax.text(time.max() + 0.2, thresholds.theta_2_high, r"$\theta_{2,high}$", va="center", fontsize=style.tick_size)

    # Crossing points
    idx_cross = np.where(np.diff((p >= thresholds.theta_1).astype(int)) != 0)[0]
    cross_scatter = ax.scatter([], [], s=40, color=style.uncertain_color, zorder=5)

    # Axes formatting
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlim(time.min(), time.max())
    ax.set_ylabel("Pain probability", fontsize=style.label_size)
    ax.set_xlabel("Time (s)", fontsize=style.label_size)
    if has_regions:
        ax.set_xlabel("")
        ax.tick_params(labelbottom=False)
    ax.tick_params(axis="both", labelsize=style.tick_size)
    ax.grid(True, axis="y", alpha=0.18, color=style.grid_color)
    ax.grid(False, axis="x")

    label_txt = "Pain" if true_label == 1 else "No pain"
    pred_txt = "Pain" if ps.pred_label == 1 else "No pain"
    reliability_parts = [f"Mean p?={ps.p_summary:.3f} ? {pred_txt}"]
    if ps.sigma_summary is not None:
        reliability_parts.append(
            f"Mean ??={ps.sigma_summary:.3f} ? {'Certain' if ps.sigma_summary <= thresholds.theta_3 else 'Uncertain'}"
        )
    reliability_parts.append(f"Crossings={int(idx_cross.size)}")

    ax.set_title(
        f"{video_name} | Model: {model_name} | True: {label_txt} | " + " | ".join(reliability_parts),
        fontsize=style.title_size,
        pad=10,
    )

    handles = [
        plt.Line2D([0], [0], color=style.certain_color, lw=2.2, label="Probability (certain)"),
        plt.Line2D([0], [0], color=style.uncertain_color, lw=2.6, label="Probability (uncertain)"),
    ]
    if sigma is not None:
        handles.append(plt.Rectangle((0, 0), 1, 1, color=style.ci_color, alpha=style.ci_alpha, label="?? band"))
    ax.legend(handles=handles, loc="upper left", frameon=True, framealpha=0.9)

    # Region curves
    if has_regions:
        axr = fig.add_subplot(gs[1], sharex=ax)
        region_cols = [c for c in region_df.columns if c not in ("frame", "frame_idx", "time_s")]
        if "time_s" in region_df.columns:
            time_r = region_df["time_s"].to_numpy(dtype=float)
        elif "frame_idx" in region_df.columns:
            time_r = region_df["frame_idx"].to_numpy(dtype=float)
        else:
            time_r = np.arange(len(region_df), dtype=float)

        if region_cols:
            if region_selection == "max":
                agg = region_df[region_cols].max(axis=0)
            else:
                agg = region_df[region_cols].mean(axis=0)
            agg = agg[agg > 0]

            if region_top_k is None or region_top_k <= 0:
                top_regions = agg.sort_values(ascending=False).index.tolist()
            else:
                top_regions = agg.sort_values(ascending=False).head(region_top_k).index.tolist()

            if top_regions:
                data = region_df[top_regions].to_numpy(dtype=float)
                if region_smooth_window > 1:
                    data = np.column_stack([
                        _moving_average_1d(data[:, i], region_smooth_window) for i in range(data.shape[1])
                    ])
                default_colors = plt.cm.get_cmap("tab20", len(top_regions))
                region_lines = []
                for idx, region in enumerate(top_regions):
                    color = REGION_COLOR_MAP.get(region, default_colors(idx))
                    line, = axr.plot([], [], lw=1.8, color=color, label=region)
                    region_lines.append(line)
                axr.legend(loc="upper right", ncol=2, fontsize=9)
            else:
                top_regions = []
                data = None
                region_lines = []
                axr.text(0.5, 0.5, "No positive region scores", transform=axr.transAxes, ha="center", va="center")
        else:
            top_regions = []
            data = None
            region_lines = []
            axr.text(0.5, 0.5, "No region columns to plot", transform=axr.transAxes, ha="center", va="center")

        axr.set_ylabel("Region importance", fontsize=style.label_size)
        axr.set_xlabel("Time (s)", fontsize=style.label_size)
        axr.tick_params(axis="both", labelsize=style.tick_size)
        axr.grid(True, axis="y", alpha=0.18, color=style.grid_color)
        axr.set_ylim(-0.05, 1.05)
        if time_r.size:
            axr.set_xlim(time.min(), time.max())
    else:
        axr = None
        time_r = None
        data = None
        region_lines = []

    # Frame + XAI panel
    if has_regions:
        ax_strip = fig.add_subplot(gs[2])
    else:
        ax_strip = fig.add_subplot(gs[1])

    h, w = out_size[1], out_size[0]
    strip_init = np.zeros((h * 2, w, 3), dtype=np.float32)
    strip_im = ax_strip.imshow(strip_init)
    ax_strip.axis("off")
    ax_strip.text(-0.01, 0.75, "Frames", transform=ax_strip.transAxes, rotation=90, va="center", ha="right", fontsize=style.label_size)
    ax_strip.text(-0.01, 0.25, "Frames + XAI", transform=ax_strip.transAxes, rotation=90, va="center", ha="right", fontsize=style.label_size)

    # Video writer setup
    fig.canvas.draw()
    width, height = fig.canvas.get_width_height()
    pad_w = width % 2
    pad_h = height % 2
    # Defensive: ensure fps_out exists even if this cell is run with an older definition in memory
    if "fps_out" not in locals():
        fps_out = max(1.0, float(fps) / float(frame_step))

    fourcc = cv2.VideoWriter_fourcc(*codec)
    writer = cv2.VideoWriter(str(out_path), fourcc, float(fps_out), (width + pad_w, height + pad_h))

    frame_indices = list(range(0, len(img_files), frame_step))

    if show_progress:
        iterator = tqdm(frame_indices, desc=f"Animating {video_name}")
    else:
        iterator = frame_indices

    merged_dir = xai_root / video_dir.name / "MERGED_MASKS"
    fill_between = None

    def _set_line(line, x, y):
        n = min(len(x), len(y))
        if n <= 0:
            line.set_data([], [])
        else:
            line.set_data(x[:n], y[:n])


    for k, frame_idx in enumerate(iterator):
        frame_path = img_files[frame_idx]

        frame = _safe_read_rgb(frame_path, out_size)
        mask_path = merged_dir / f"{frame_path.stem}.npz"
        mask = _safe_read_XAI_mask(mask_path, out_size)
        alpha = mask.max(axis=2, keepdims=True)
        overlay = frame * (1.0 - alpha) + mask
        strip = np.vstack([frame, np.clip(overlay, 0.0, 1.0)])
        strip_im.set_data(strip)

        t_now = frame_idx / float(fps)
        curve_idx = int(np.searchsorted(time, t_now, side="right"))

        _set_line(line_certain, time[:curve_idx], p_certain[:curve_idx])
        _set_line(line_uncertain, time[:curve_idx], p_uncertain[:curve_idx])

        if sigma is not None:
            if fill_between is not None:
                fill_between.remove()
            upper = np.clip(p[:curve_idx] + sigma[:curve_idx], 0, 1)
            lower = np.clip(p[:curve_idx] - sigma[:curve_idx], 0, 1)
            fill_between = ax.fill_between(time[:curve_idx], lower, upper, color=style.ci_color, alpha=style.ci_alpha, edgecolor="none")

        if idx_cross.size:
            valid_cross = idx_cross[time[idx_cross] <= t_now]
            if valid_cross.size:
                cross_scatter.set_offsets(np.c_[time[valid_cross], np.full(valid_cross.shape, thresholds.theta_1)])
            else:
                cross_scatter.set_offsets(np.empty((0, 2)))

        if has_regions and data is not None and time_r is not None:
            region_idx = int(np.searchsorted(time_r, t_now, side="right"))
            for j, line in enumerate(region_lines):
                _set_line(line, time_r[:region_idx], data[:region_idx, j])

        fig.canvas.draw()
        # Matplotlib compatibility: newer backends expose buffer_rgba() instead of tostring_rgb()
        if hasattr(fig.canvas, "tostring_rgb"):
            img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
            img = img.reshape((height, width, 3))
        else:
            buf = np.asarray(fig.canvas.buffer_rgba())
            img = buf[..., :3].copy()
        if pad_w or pad_h:
            img = cv2.copyMakeBorder(img, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=(255, 255, 255))
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        writer.write(img_bgr)

    writer.release()
    plt.close(fig)

    return out_path








In [ ]:
video_name = list(results_video.keys())[0]
out_path = render_pain_sign_animation(
    video_name=video_name,
    video_results=results_video[video_name],
    model_name=model_name,
    video_dir=Path(path_icopevid_frames) / video_name,
    xai_root=Path(path_experiments) / "icopevid" / "XAI",
    out_path=r"C:\Users\leonardo\Desktop\icopevid_results\pain_sign_anim.mp4",
    mcdp=mcdp,
    fps=30,                # set to your real FPS if known
    duration_s=20.0,       # keep consistent with your plots
    frame_step=1,          # >1 will skip frames but keep real-time speed
    include_region_curves=True,
    region_top_k=-1,
)
print(out_path)


In [ ]:


# ---------------------------------------------------------------------
# Usage (example)
# ---------------------------------------------------------------------
plt.style.use('utils\plotstyle.mplstyle')#
xai_root = Path(path_experiments) / "icopevid" / "XAI"

arrays = run_pain_sign_report(
     results_video=results_video,
     model_name=model_name,
     path_icopevid_frames=path_icopevid_frames,
     xai_root=xai_root,
     out_dir=r"C:\Users\leonardo\Desktop\icopevid_results",
     mcdp=mcdp,
     ma_window=30,
     duration_s=20.0,
     frame_step=30,
     region_frame_step=1,
     include_region_curves=True,
     region_top_k=-1,
     region_selection="mean",
     region_smooth_window=30,
)

print(validation_metrics(arrays["preds"], arrays["probs"], arrays["labels"]))


Frames S004_Pain_1_[0]_20s:  46%|████▌     | 277/600 [00:33<00:41,  7.75it/s]